In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, f1_score
import warnings
warnings.filterwarnings('ignore')

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" Device: {device}")

 Device: cuda


In [2]:
# Load data
train_df = pd.read_csv('C:/Users/Riyad/projects/fake_news/train_final.csv')
val_df = pd.read_csv('C:/Users/Riyad/projects/fake_news/val_final.csv')
test_df = pd.read_csv('C:/Users/Riyad/projects/fake_news/test_final.csv')

# Combine headline + content
train_df['text'] = train_df['headline'].fillna('') + ' ' + train_df['content'].fillna('')
val_df['text'] = val_df['headline'].fillna('') + ' ' + val_df['content'].fillna('')
test_df['text'] = test_df['headline'].fillna('') + ' ' + test_df['content'].fillna('')

# Label encoding
label2id = {'authentic': 0, 'fake': 1, 'ai_fake': 2}
train_df['label_id'] = train_df['label'].map(label2id)
val_df['label_id'] = val_df['label'].map(label2id)
test_df['label_id'] = test_df['label'].map(label2id)

# Remove NaN
train_df = train_df.dropna(subset=['text', 'label_id'])
val_df = val_df.dropna(subset=['text', 'label_id'])
test_df = test_df.dropna(subset=['text', 'label_id'])

print(" Data loaded!")
print(f"Train: {len(train_df)}")
print(f"Val:   {len(val_df)}")
print(f"Test:  {len(test_df)}")

 Data loaded!
Train: 10500
Val:   2250
Test:  2250


In [4]:
from collections import Counter

# Build vocabulary from training data
def tokenize(text):
    return text.split()

# Count word frequencies
word_freq = Counter()
for text in train_df['text']:
    word_freq.update(tokenize(str(text)))

# Build vocab (top 30000 words)
vocab_size = 30000
vocab = ['<PAD>', '<UNK>'] + [w for w, _ in word_freq.most_common(vocab_size - 2)]
word2idx = {w: i for i, w in enumerate(vocab)}

print(f"Vocabulary built!")
print(f"Vocab size: {len(vocab)}")

# Dataset class
MAX_LEN = 256

class BiLSTMDataset(Dataset):
    def __init__(self, texts, labels, word2idx, max_len=MAX_LEN):
        self.texts = texts
        self.labels = labels
        self.word2idx = word2idx
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        tokens = tokenize(str(self.texts[idx]))[:self.max_len]
        ids = [self.word2idx.get(t, 1) for t in tokens]
        
        # Padding
        if len(ids) < self.max_len:
            ids += [0] * (self.max_len - len(ids))
        
        return {
            'input_ids': torch.tensor(ids, dtype=torch.long),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

# Create datasets
train_dataset = BiLSTMDataset(train_df['text'].values, train_df['label_id'].values, word2idx)
val_dataset = BiLSTMDataset(val_df['text'].values, val_df['label_id'].values, word2idx)
test_dataset = BiLSTMDataset(test_df['text'].values, test_df['label_id'].values, word2idx)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)
test_loader = DataLoader(test_dataset, batch_size=32)

print(f"Train batches: {len(train_loader)}")

Vocabulary built!
Vocab size: 30000
Train batches: 329


In [5]:
# BiLSTM Model
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, num_classes, dropout=0.3):
        super(BiLSTMClassifier, self).__init__()
        
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        output, (hidden, _) = self.lstm(embedded)
        
        # Concat forward and backward hidden states
        hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
        hidden = self.dropout(hidden)
        return self.fc(hidden)

# Initialize model
model = BiLSTMClassifier(
    vocab_size=len(vocab),
    embed_dim=128,
    hidden_dim=256,
    num_layers=2,
    num_classes=3,
    dropout=0.3
).to(device)

print(" BiLSTM Model ready!")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

# Optimizer and loss
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

 BiLSTM Model ready!
Total parameters: 6,209,027


In [6]:
# Training function
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    
    for batch in loader:
        optimizer.zero_grad()
        
        input_ids = batch['input_ids'].to(device)
        labels = batch['label'].to(device)
        
        outputs = model(input_ids)
        loss = criterion(outputs, labels)
        total_loss += loss.item()
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
    
    return total_loss / len(loader)

# Evaluation function
def evaluate(model, loader):
    model.eval()
    preds = []
    true_labels = []
    
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            labels = batch['label'].to(device)
            
            outputs = model(input_ids)
            pred = outputs.argmax(dim=1)
            
            preds.extend(pred.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())
    
    return preds, true_labels

# Train for 5 epochs
print("Training started!")
best_f1 = 0

for epoch in range(5):
    print(f"\nEpoch {epoch+1}/5")
    
    train_loss = train_epoch(model, train_loader, optimizer, criterion)
    print(f"Train Loss: {train_loss:.4f}")
    
    val_preds, val_true = evaluate(model, val_loader)
    val_f1 = f1_score(val_true, val_preds, average='macro')
    print(f"Val Macro F1: {val_f1*100:.2f}%")
    
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(
            model.state_dict(),
            'C:/Users/Riyad/projects/fake_news/bilstm_best.pt'
        )
        print(f" Best model saved!")

print(f"\nBest Val F1: {best_f1*100:.2f}%")

Training started!

Epoch 1/5
Train Loss: 0.8533
Val Macro F1: 53.57%
 Best model saved!

Epoch 2/5
Train Loss: 0.5234
Val Macro F1: 82.63%
 Best model saved!

Epoch 3/5
Train Loss: 0.3872
Val Macro F1: 87.41%
 Best model saved!

Epoch 4/5
Train Loss: 0.2943
Val Macro F1: 88.17%
 Best model saved!

Epoch 5/5
Train Loss: 0.2281
Val Macro F1: 88.57%
 Best model saved!

Best Val F1: 88.57%


In [7]:
# Load best model
model.load_state_dict(torch.load(
    'C:/Users/Riyad/projects/fake_news/bilstm_best.pt'
))

# Test evaluation
test_preds, test_true = evaluate(model, test_loader)

print("=== BiLSTM Test Results ===")
print(classification_report(
    test_true,
    test_preds,
    target_names=['authentic', 'fake', 'ai_fake']
))

test_f1 = f1_score(test_true, test_preds, average='macro')
print(f"Test Macro F1: {test_f1*100:.2f}%")

=== BiLSTM Test Results ===
              precision    recall  f1-score   support

   authentic       0.80      0.89      0.84       750
        fake       0.88      0.75      0.81       750
     ai_fake       0.95      0.98      0.97       750

    accuracy                           0.87      2250
   macro avg       0.88      0.87      0.87      2250
weighted avg       0.88      0.87      0.87      2250

Test Macro F1: 87.29%
